# Hardware input-calibration plotter

Compares FC telemetry (MAVSDK IMU: `Angular Velocity FRD`, `Acceleration`) against the commanded body-rate + thrust profile sent by `record_input_calibration.py` on the real Pi hardware.

Modeled after `PX4_Gazebo/notebooks/plotter_input_calibration.ipynb`, but **hardware has no ground truth for input calibration** (unlike the SITL/Gazebo version) -- `record_input_calibration.py`'s own docstring notes achieved rates come from `FC_node.getLogData()` (real IMU/EKF telemetry), which is exactly what input calibration is meant to characterize against the COMMAND. So this notebook only has the SITL notebook's §2 ("Input transfer: commanded vs achieved" -- the headline section) with no §1/§3 GT-diagnostic sections, since those need Gazebo pose ground truth that doesn't exist here.

**Battery-voltage caveat (read before trusting any run):** `HW_HOVER_THROTTLE_NORM` is only valid in the ~22.4-24.0V battery range (see memory `project_hover_voltage_curve` / `project_hover_throttle_search_2026_07_09`). A run recorded outside that range will show corrupted/noisy correlations from a systematic climb or sink bias, not real rate-tracking noise -- this notebook prints the battery voltage prominently for exactly that reason.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

sys.path.insert(0, os.path.abspath('..'))
from analyze_input_calibration import per_run_metrics, mad_trimmed_mean, AXES

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Keep these in sync with record_input_calibration.py / hardware_landing.py --
# NOT auto-imported since that module has hardware-only imports (mavsdk FC,
# Controller) that may not be importable off the Pi.
MASS_KG = 1.230                 # confirmed total flight mass (airframe+props 0.965 + battery 0.265)
HOVER_THROTTLE_NORM = 0.388      # confirmed, valid ~22.4-24.0V only
THRUST_SLOPE_N_PER_UNIT = 34.8   # derived 2026-07-10 from wide +/-2.5N excitation (was 42.3 SITL placeholder)
G = 9.81

In [ ]:
# Pick a run directory.
#  - Default: most recent run in Test_Data/Calibration/Input/ (RUN_INDEX = 0).
#  - Set RUN_INDEX = 1 for the previous run, 2 for the one before, ...
#  - Prints battery voltage + sample count for every candidate so a
#    corrupted/partial/low-battery run is visible before you pick it.
CAL_DIR = os.path.join('..', 'Test_Data', 'Calibration', 'Input')
RUN_INDEX = 0
N_SHOW = 10

runs = sorted(
    (d for d in os.listdir(CAL_DIR) if os.path.isdir(os.path.join(CAL_DIR, d))),
    key=lambda d: os.path.getmtime(os.path.join(CAL_DIR, d)),
    reverse=True,
)

print(f'{"idx":>3s}  {"run":32s}  {"n_cmd":>6s}  {"battery":>8s}')
for i, d in enumerate(runs[:N_SHOW]):
    gt = np.load(os.path.join(CAL_DIR, d, 'Ground_Truth.npy'), allow_pickle=True).item()
    cmd = np.array(gt.get('Command', []))
    batt = gt.get('Battery Voltage')
    batt_str = f'{batt:.2f}V' if batt else 'n/a'
    marker = '  <-- selected' if i == RUN_INDEX else ''
    print(f'{i:>3d}  {d:32s}  {len(cmd):>6d}  {batt_str:>8s}{marker}')

run_dir = os.path.join(CAL_DIR, runs[RUN_INDEX])
print(f'\nLoading: {run_dir}')

## §1 — Data preparation

Load telemetry (IMU angular velocity + acceleration, body-FRD) and the commanded rate/thrust profile, aligned to a common timeline.

In [ ]:
tel = np.load(os.path.join(run_dir, 'Telemetry_Data.npy'), allow_pickle=True).item()
gt = np.load(os.path.join(run_dir, 'Ground_Truth.npy'), allow_pickle=True).item()

battery_voltage = gt.get('Battery Voltage')
print(f"Battery voltage: {battery_voltage:.2f}V" if battery_voltage else 'Battery voltage: NOT LOGGED (older run)')
if battery_voltage is not None and not (22.4 <= battery_voltage <= 24.0):
    print('*** WARNING: battery voltage outside the confirmed ~22.4-24.0V HOVER_THROTTLE_NORM=0.388 plateau. ***')
    print('*** Correlations/gains below may be corrupted by a systematic climb/sink bias, not real rate noise. ***')

t_cmd = np.array(gt['Time'])
cmd = np.array(gt['Command'])
nm = min(len(t_cmd), len(cmd))
t_cmd, cmd = t_cmd[-nm:], cmd[-nm:]
w_u = cmd[:, :3]   # commanded rate (rad/s), body-FRD
B_T_u = cmd[:, 3]  # commanded thrust delta (N, excess-over-hover)

print(f'\nn_cmd_samples = {nm},  profile duration = {t_cmd[-1] - t_cmd[0]:.2f}s')
print(f'unique commands:\n{np.unique(np.round(cmd, 3), axis=0)}')

t_imu = np.array(tel['IMU Timestamp']) - gt['Start Time']
w_t = np.array([[a.forward_rad_s, a.right_rad_s, a.down_rad_s]
                for a in tel['Angular Velocity FRD']])
a_t = np.array([[a.forward_m_s2, a.right_m_s2, a.down_m_s2]
                for a in tel['Acceleration']])
mask = (t_imu >= t_cmd[0]) & (t_imu <= t_cmd[-1])
t_imu, w_t, a_t = t_imu[mask], w_t[mask], a_t[mask]
print(f'n_imu_samples (within command window) = {mask.sum()}')

## §2 — Command vs achieved: body angular rate

Rate-loop tracking check: how well does PX4's inner loop follow the commanded body-FRD rate sent by `record_input_calibration.py`? The measured trace (PX4 IMU gyro) should lag the command by the rate-loop response time; a flat/uncorrelated measured trace (as currently seen on pitch) indicates the commanded rate isn't producing a resolvable response above noise at this amplitude.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['x (roll)', 'y (pitch)', 'z (yaw)'])):
    ax.plot(t_imu, w_t[:, i], label=r'$w_{\mathrm{meas}}$ (PX4 IMU)')
    ax.plot(t_cmd, w_u[:, i], label=r'$w_{\mathrm{cmd}}$ (commanded)',
            linestyle='--', linewidth=2)
    ax.set(title=f'Body angular rate $w_{{{name}}}$ (body-FRD, rad/s) — cmd vs IMU',
           xlabel='t (s)', ylabel='rad/s')
    ax.legend()
plt.show()

## §3 — Command vs achieved: thrust (via IMU specific force)

Thrust acts along body −z, so `a_down` (accelerometer specific force) ≈ −Thrust/mass. Expected `a_down` from the commanded `B_T` uses the model `a_down = -F_hover/mass + B_T·SLOPE_true/(SLOPE_assumed·mass)` (see `project_hover_throttle_search_2026_07_09` memory for the derivation) -- overlaying measured vs this model-predicted trace is a direct check on whether `THRUST_SLOPE_N_PER_UNIT` is still a good fit for this run's battery state.

In [ ]:
a_down_meas = a_t[:, 2]
a_down_hover_baseline = a_down_meas[np.abs(np.interp(t_imu, t_cmd, B_T_u)) < 1e-6].mean() if (np.abs(B_T_u) < 1e-6).any() else -G

B_T_on_imu = np.interp(t_imu, t_cmd, B_T_u)
a_down_expected = a_down_hover_baseline + B_T_on_imu * THRUST_SLOPE_N_PER_UNIT / (42.3 * MASS_KG)

plt.figure(figsize=(12, 5), constrained_layout=True)
plt.plot(t_imu, a_down_meas, label=r'$a_{down}$ measured (IMU)')
plt.plot(t_imu, a_down_expected, label=r'$a_{down}$ expected (from commanded $B_T$ + derived slope)', linestyle='--')
plt.axhline(a_down_hover_baseline, color='grey', linestyle=':', alpha=0.6,
            label=f'hover baseline = {a_down_hover_baseline:.2f} m/s²')
plt.title('Body thrust (specific force, m/s²) — commanded vs achieved')
plt.xlabel('t (s)'); plt.ylabel('a_down (m/s²)')
plt.legend()
plt.show()

## §4 — Per-axis command→response quality (this run)

Reuses `analyze_input_calibration.py::per_run_metrics` (single source of truth -- no duplicated logic) for Pearson r, cross-correlation lag, and gain per axis.

In [ ]:
m = per_run_metrics(run_dir)
if m is None:
    print('Insufficient samples in this run for per_run_metrics (too short / aborted too early).')
else:
    print(f'{"axis":4s}  {"r":>6s}  {"lag_ms":>7s}  {"gain":>7s}')
    for ax in AXES:
        print(f'{ax:4s}  {m[ax]["r"]:6.3f}  {m[ax]["lag_ms"]:7.1f}  {m[ax]["gain"]:7.3f}')
    print()
    print('  r close to 1   -> FC closely tracks cmd (good)')
    print('  r much < 1     -> FC response dominated by disturbance rejection / noise')
    print('  lag_ms         -> rate-loop deadtime (MAVSDK transit + PX4 inner loop)')
    print('  gain ~= 1.0    -> FC matches cmd amplitude; << 1 = underdamped tracking')

## §5 — Aggregate across all valid-battery runs

Same MAD-trimmed aggregation as `analyze_input_calibration.py`, but filtered here to only runs whose logged battery voltage falls in the confirmed 22.4-24.0V plateau -- excludes the corrupted low-battery batch and any run predating battery-voltage logging (`batt=n/a`).

In [ ]:
VOLTAGE_MIN, VOLTAGE_MAX = 22.4, 24.0

valid_runs = []
for d in runs:
    gt_d = np.load(os.path.join(CAL_DIR, d, 'Ground_Truth.npy'), allow_pickle=True).item()
    v = gt_d.get('Battery Voltage')
    if v is not None and VOLTAGE_MIN <= v <= VOLTAGE_MAX:
        valid_runs.append((d, v))

print(f'{len(valid_runs)} valid-battery runs (of {len(runs)} total):')
for d, v in valid_runs:
    print(f'  {d}  ({v:.2f}V)')

all_metrics = {ax: dict(r=[], lag_ms=[], gain=[]) for ax in AXES}
for d, v in valid_runs:
    mm = per_run_metrics(os.path.join(CAL_DIR, d))
    if mm is None:
        continue
    for ax in AXES:
        for key in ('r', 'lag_ms', 'gain'):
            all_metrics[ax][key].append(mm[ax][key])

print(f'\n{"axis":4s}  {"r":>10s}  {"lag_ms":>11s}  {"gain":>10s}')
for ax in AXES:
    r_tm, r_n, _ = mad_trimmed_mean(all_metrics[ax]['r'])
    l_tm, l_n, _ = mad_trimmed_mean(all_metrics[ax]['lag_ms'])
    g_tm, g_n, _ = mad_trimmed_mean(all_metrics[ax]['gain'])
    print(f'{ax:4s}  {r_tm:6.3f} (n={r_n})  {l_tm:7.1f} (n={l_n})  {g_tm:7.3f} (n={g_n})')